# UniLex AI — Continue Synthetic Generation

### الهدف
إكمال `student_friendly_explanation` و `example` للمصطلحات الناقصة في ملف UniLex AI (3,000 term).

### مهم
- الـ922 مصطلح التي تم توليدها سابقًا **لن يعاد توليدها**.
- يتم الحفظ تلقائيًا أثناء التشغيل في `UniLex_AI_3000_progress.csv`.
- إذا أوقفتي التشغيل، أعيدي تشغيل الخلايا من البداية؛ الكود سيقرأ ملف الـprogress ويكمل من آخر نقطة محفوظة.
- **الأفضل تنزيل ملف الـprogress إلى جهازك** عند إيقاف Colab أو انتهاء الجلسة، لأن ملفات `/content` قد تختفي عند انتهاء الـruntime.

## 1. Install libraries

In [3]:
!pip install -q transformers accelerate bitsandbytes pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.4 MB/s eta 0:00:00


## 2. Imports and GPU check

In [4]:
import os
import re
import json
import time
import torch
import pandas as pd
from tqdm.auto import tqdm
from google.colab import files

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: GPU is not enabled. Enable a GPU in Colab before running the model.')

CUDA available: True
GPU: Tesla T4


## 3. Upload the 3,000-term dataset

Upload `UniLex_AI_3000_selected.csv`.

If `UniLex_AI_3000_progress.csv` already exists in the Colab session, the notebook will automatically use it instead.

In [ ]:
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [5]:
INPUT_FILE = 'UniLex_AI_2000_selected.csv'
PROGRESS_FILE = 'UniLex_AI_3000_progress.csv'

if os.path.exists(PROGRESS_FILE):
    INPUT_FILE = PROGRESS_FILE
    print('Existing progress found. Resuming from:', PROGRESS_FILE)
elif os.path.exists('UniLex_AI_2000_selected.csv'):
    print('Starting from:', INPUT_FILE)
else:
    csv_files = [f for f in os.listdir('/content') if f.endswith('.csv')]
    raise FileNotFoundError(
        'Please upload UniLex_AI_3000_selected.csv. Found: ' + str(csv_files)
    )

df = pd.read_csv(INPUT_FILE).fillna('')

print('Rows:', len(df))
print('Columns:', list(df.columns))

Existing progress found. Resuming from: UniLex_AI_3000_progress.csv
Rows: 1922
Columns: ['id', 'term', 'definition', 'category', 'difficulty', 'related_terms', 'student_friendly_explanation', 'example']


## 4. Prepare required columns

In [6]:
required_columns = [
    'term',
    'definition',
    'category',
    'difficulty',
    'related_terms',
    'student_friendly_explanation',
    'example'
]

for col in required_columns:
    if col not in df.columns:
        df[col] = ''

df = df.fillna('')

print('Dataset prepared.')

Dataset prepared.


## 5. Load Qwen2.5-3B-Instruct

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto'
)

model.eval()

print('Model loaded:', MODEL_NAME)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-3B-Instruct


## 6. Prompt

In [8]:
def build_prompt(row):
    return f'''You are helping build UniLex, a student-friendly academic dictionary for university students studying Artificial Intelligence.

Term:
{row['term']}

Formal definition:
{row['definition']}

Category:
{row['category']}

Related terms:
{row['related_terms']}

Generate ONLY valid JSON with exactly these two fields:
{{
  "student_friendly_explanation": "...",
  "example": "..."
}}

Requirements:
- Explain the term in simple English suitable for a university student.
- Keep the explanation accurate and faithful to the formal definition.
- Keep each field to 1–2 sentences.
- Do not invent facts or give the term abilities that are not in its definition.
- The example must demonstrate how the concept actually works, not just mention a related application.
- For mathematical functions, describe the transformation they apply; do not claim they choose important features unless the definition says so.
- Avoid unnecessary technical complexity.
- Return JSON only.'''


## 7. Generate one term

In [21]:
def generate_one(row):
    prompt = build_prompt(row) + """
Additional requirements:
- Return valid JSON with plain-text values.
- Do not use backslashes or LaTeX.
- Keep each field to one or two sentences.
- Use an accurate, concrete example.
- Do not claim that approximations always preserve accuracy.
"""

    if str(row['term']).strip().lower() == 'hard swish':
        prompt = """
Write a short dictionary entry using ONLY these verified facts:
Hard Swish approximates the Swish activation function with
simpler computations, useful in mobile neural networks.
An input of -4 produces 0.
An input of 4 produces 4.

Return ONLY valid JSON with exactly these two keys:
"student_friendly_explanation": one simple sentence about its purpose.
"example": one sentence showing the two input-output pairs above.

Use plain English. Do not include formulas, LaTeX, backslashes,
or any additional claims.
"""

    messages = [{'role': 'user', 'content': prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=350,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        output[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    print("MODEL RESPONSE:")
    print(response)

    match = re.search(r'\{.*\}', response, re.DOTALL)

    if not match:
        print("ERROR: No complete JSON object found.")
        return {}

    try:
        result = json.loads(match.group())
    except json.JSONDecodeError as error:
        print("JSON ERROR:", error)
        return {}

    required = ['student_friendly_explanation', 'example']

    if isinstance(result, dict) and all(
        isinstance(result.get(key), str)
        and result[key].strip()
        for key in required
    ):
        return result

    print("ERROR: Required fields are missing or empty.")
    return {}


## 8. Test one row before starting

In [22]:
test_candidates = [
    i for i in df.index
    if str(df.loc[i, 'student_friendly_explanation']).strip() == ''
    or str(df.loc[i, 'example']).strip() == ''
]

if test_candidates:
    test_idx = test_candidates[0]
    test_result = generate_one(df.loc[test_idx])
    print('Term:', df.loc[test_idx, 'term'])
    print(json.dumps(test_result, ensure_ascii=False, indent=2))
    print("ORIGINAL DEFINITION:", df.loc[test_idx, 'definition'])
else:
    print('No missing rows. Dataset is already complete.')

MODEL RESPONSE:
{
    "student_friendly_explanation": "The Hard Swish function simplifies the Swish activation for mobile networks, making it easier to compute.",
    "example": "For an input of -4, the output is 0, and for an input of 4, the output is 4."
}
Term: Hard Swish
{
  "student_friendly_explanation": "The Hard Swish function simplifies the Swish activation for mobile networks, making it easier to compute.",
  "example": "For an input of -4, the output is 0, and for an input of 4, the output is 4."
}
ORIGINAL DEFINITION: A piecewise linear approximation of Swish used in mobile architectures.


## 9. Identify only the missing terms
The already-generated 922 terms are skipped automatically.

In [23]:
def needs_generation(row):
    explanation = str(row['student_friendly_explanation']).strip()
    example = str(row['example']).strip()
    return explanation == '' or example == ''

pending_indices = [
    i for i in df.index
    if needs_generation(df.loc[i])
]

completed = len(df) - len(pending_indices)

print('Total terms:', len(df))
print('Already completed:', completed)
print('Need generation:', len(pending_indices))

Total terms: 1922
Already completed: 1918
Need generation: 4


## 10. Full generation + automatic checkpoint saving

**Important:** The file is saved every 10 processed terms. If you press Stop, all completed batches up to the latest save remain in `UniLex_AI_3000_progress.csv`.

In [24]:
SAVE_EVERY = 10
start_time = time.time()
processed = 0
partial_results = 0

try:
    for idx in tqdm(pending_indices, desc='Generating'):
        result = generate_one(df.loc[idx])

        explanation = result.get('student_friendly_explanation', '')
        example = result.get('example', '')

        if isinstance(explanation, str) and explanation.strip():
            df.at[idx, 'student_friendly_explanation'] = explanation.strip()

        if isinstance(example, str) and example.strip():
            df.at[idx, 'example'] = example.strip()

        if not (explanation and example):
            partial_results += 1

        processed += 1

        if processed % SAVE_EVERY == 0:
            df.to_csv(
                PROGRESS_FILE,
                index=False,
                encoding='utf-8-sig'
            )

            elapsed = time.time() - start_time
            rate = processed / elapsed if elapsed > 0 else 0
            remaining = len(pending_indices) - processed
            eta = remaining / rate if rate > 0 else 0

            print(
                f'Checkpoint saved | {processed}/{len(pending_indices)} | '
                f'ETA: {eta/3600:.2f} hours'
            )

except KeyboardInterrupt:
    print('\nSTOPPED BY USER — saving progress now...')
    df.to_csv(
        PROGRESS_FILE,
        index=False,
        encoding='utf-8-sig'
    )
    print('Progress saved:', PROGRESS_FILE)

except Exception as e:
    print('\nERROR — saving progress before stopping...')
    df.to_csv(
        PROGRESS_FILE,
        index=False,
        encoding='utf-8-sig'
    )
    print('Progress saved:', PROGRESS_FILE)
    raise e

else:
    df.to_csv(
        PROGRESS_FILE,
        index=False,
        encoding='utf-8-sig'
    )
    print('Generation finished successfully.')
    print('Final progress file:', PROGRESS_FILE)

Generating:   0%|          | 0/4 [00:00<?, ?it/s]

MODEL RESPONSE:
{
    "student_friendly_explanation": "The Hard Swish function simplifies the Swish activation for mobile networks, making it easier to compute.",
    "example": "For an input of -4, the output is 0, and for an input of 4, the output is 4."
}
MODEL RESPONSE:
{
  "student_friendly_explanation": "Normalization Form is a standardized version of Unicode characters that ensures visually similar text can be compared as equal, making it easier to process and compare text data.",
  "example": "In normalization form KD, the sequence of characters 'á' and 'a\u0308' (where \u03ml is the combining diaresis) are treated as equivalent, allowing text analysis tools to correctly identify and count them as the same character."
}
JSON ERROR: Invalid \uXXXX escape: line 3 column 94 (char 311)
MODEL RESPONSE:
{
  "student_friendly_explanation": "Imagine you're trying to find the lowest point in a hilly landscape. The objective landscape is like that whole area of hills and valleys where yo

## 11. Check the result

In [26]:
missing_explanations = (df['student_friendly_explanation'].astype(str).str.strip() == '').sum()
missing_examples = (df['example'].astype(str).str.strip() == '').sum()

print('Total rows:', len(df))
print('Missing explanations:', missing_explanations)
print('Missing examples:', missing_examples)
print('Completed rows:', len(df) - max(missing_explanations, missing_examples))
for term in ['Normalization Form', 'Polynomial Kernel']:
    print("\nTERM:", term)
    for definition in df.loc[df['term'].eq(term), 'definition']:
        print("DEFINITION:", definition)

Total rows: 1922
Missing explanations: 1
Missing examples: 1
Completed rows: 1921

TERM: Normalization Form
DEFINITION: A canonical Unicode representation chosen so visually identical text compares equal.

TERM: Polynomial Kernel
DEFINITION: A kernel corresponding to all feature products up to a chosen degree.


In [27]:
repair_facts = {
    "Normalization Form": """
Unicode normalization gives equivalent character sequences
a consistent representation.
Example: NFC converts a precomposed a with an acute accent
and a separate a followed by a combining acute accent
to the same representation.
It does not make all visually similar characters equivalent.
Describe the example in words, without Unicode codes.
""",
    "Polynomial Kernel": """
A polynomial kernel compares two inputs using polynomial
feature interactions without explicitly constructing
the expanded feature vectors.
Example: with degree 2, gamma 1, and coef0 1,
a dot product of 2 gives a kernel value of 9:
add 1 to 2, then square the result.
Use this numerical example without adding other terms.
"""
}

repair_results = {}

for term, facts in repair_facts.items():
    prompt = f"""
Write a student-friendly dictionary entry for {term}.
Use ONLY these verified facts:
{facts}

Return ONLY valid JSON with exactly two keys:
"student_friendly_explanation" and "example".
Each value must be one short sentence.
Use plain English, no LaTeX, no backslashes.
"""
    text = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt}],
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(
        text, return_tensors='pt'
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        output[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    print("\nTERM:", term)
    print(response)

    match = re.search(r'\{.*\}', response, re.DOTALL)
    try:
        result = json.loads(match.group()) if match else {}
        keys = ['student_friendly_explanation', 'example']
        if isinstance(result, dict) and all(
            isinstance(result.get(k), str) and result[k].strip()
            for k in keys
        ):
            repair_results[term] = result
        else:
            print("ERROR: Missing or empty fields.")
    except json.JSONDecodeError as error:
        print("JSON ERROR:", error)


TERM: Normalization Form
{
  "student_friendly_explanation": "Normalization forms standardize character sequences.",
  "example": "NFC normalizes 'á' (precomposed) to the same form as 'a\u0301'."
}

TERM: Polynomial Kernel
{
   "student_friendly_explanation": "Polynomial kernel is like magic that compares things without fully growing them.",
   "example": "With degree 2, it turns 2 into 9 with a special trick."
}


In [28]:
prompt = """
Rewrite the following explanation and example in simple English.
Preserve all technical facts and parameter values.
Do not use analogies, magic, tricks, or vague wording.

Explanation:
A polynomial kernel measures similarity between two input vectors
using polynomial feature interactions, without explicitly
constructing the expanded feature vectors.

Example:
For degree 2, gamma 1, and coef0 1, two vectors whose dot product
is 2 have a kernel value of 9, calculated as (2 + 1) squared.

Return ONLY valid JSON with exactly these keys:
"student_friendly_explanation" and "example".
Use one sentence per field. No LaTeX or backslashes.
"""

text = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': prompt}],
    tokenize=False,
    add_generation_prompt=True
)
inputs = tokenizer(text, return_tensors='pt').to(model.device)

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    output[0][inputs['input_ids'].shape[1]:],
    skip_special_tokens=True
).strip()

print(response)

match = re.search(r'\{.*\}', response, re.DOTALL)
if match:
    try:
        candidate = json.loads(match.group())
        keys = ['student_friendly_explanation', 'example']
        if isinstance(candidate, dict) and all(
            isinstance(candidate.get(k), str) and candidate[k].strip()
            for k in keys
        ):
            repair_results['Polynomial Kernel'] = candidate
        else:
            print("ERROR: Missing or empty fields.")
    except json.JSONDecodeError as error:
        print("JSON ERROR:", error)
else:
    print("ERROR: No JSON found.")

"student_friendly_explanation" : A polynomial kernel compares how similar two input vectors are based on their polynomial interactions, not by fully expanding those interactions into new features.

"example" : For a degree of 2, gamma of 1, and coef0 of 1, if two vectors have a dot product of 2, their kernel value is 9, calculated as (2 + 1)².
ERROR: No JSON found.


In [29]:
match = re.search(
    r'"student_friendly_explanation"\s*:\s*(.*?)'
    r'\s*"example"\s*:\s*(.*)\Z',
    response.strip(),
    re.DOTALL
)
assert match, "Could not read the response."

repair_results['Polynomial Kernel'] = {
    'student_friendly_explanation': match.group(1).strip(),
    'example': match.group(2).strip()
}

terms = ['Normalization Form', 'Polynomial Kernel']
columns = ['student_friendly_explanation', 'example']

for term in terms:
    assert term in repair_results, f"Missing result: {term}"
    assert df['term'].eq(term).sum() == 1, f"Check term: {term}"
    for col in columns:
        assert repair_results[term][col].strip(), "Empty result."

for term in terms:
    for col in columns:
        df.loc[df['term'].eq(term), col] = repair_results[term][col]

df.to_csv(PROGRESS_FILE, index=False, encoding='utf-8-sig')

missing = df[columns].fillna('').apply(
    lambda col: col.astype(str).str.strip().eq('')
)
print("Total rows:", len(df))
print("Missing explanations:", missing[columns[0]].sum())
print("Missing examples:", missing[columns[1]].sum())
print("Completed rows:", (~missing.any(axis=1)).sum())

Total rows: 1922
Missing explanations: 0
Missing examples: 0
Completed rows: 1922


## 12. Save final file

In [30]:
FINAL_FILE = 'UniLex_AI_final_3000.csv'

df.to_csv(
    FINAL_FILE,
    index=False,
    encoding='utf-8-sig'
)

print('Saved:', FINAL_FILE)

Saved: UniLex_AI_final_3000.csv


## 13. Download the progress/final files
Run this before closing the Colab session so the files are safe on your computer.

In [31]:
if os.path.exists(PROGRESS_FILE):
    print('Downloading progress file...')
    files.download(PROGRESS_FILE)

if os.path.exists(FINAL_FILE):
    print('Downloading final file...')
    files.download(FINAL_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>